In [ ]:
# Demo to show how to do calls to the OpenAI API
# 08-16-2024
# JREG

In [ ]:
from openai import OpenAI
import os
import pandas as pd
from tqdm import tqdm
import numpy as np
from datetime import datetime
import json

In [ ]:
client = OpenAI(api_key='[ADD YOUR KEY HERE]',
               organization='[ORG ID HERE]')

os.environ['OPENAI_API_KEY'] = '[ADD YOUR KEY HERE]'

In [ ]:
# import database
df = pd.read_csv("[PATH TO FILE HERE].csv")
# show dataframe (each row will be a call)
df

In [ ]:
# check initial prompt
# This can be the information you want the LLM to summarize
df['prompt'][0]

In [ ]:
# for reproducibility
# temperature = 0, then no stochastic results
set_seed = 2024
set_temperature = 1 # default
set_maxtokens = 300 # max number of tokens used so far is 182

In [5]:
# Define a function to call the GPT API
def query_valuation(model, db, seed, temperature, max_tokens, csv_file, yes_print):
    # Initialize an empty list to store the conversation log
    conversation_log = []

    # Iterate over the DataFrame
    for index, row in db.iterrows():
        unique_id = row['UNIQUE_ID']
        # add more columns if needed 
        variable_name = row['variable_name']

        # Check if prompt are not NA for this unique_id
        if not pd.isna(row['prompt']):
            initial_prompt = row['prompt']

            full_initial_prompt = [
                {'role': 'system', 'content': 'Instructions to the LLM.'},
                {'role': 'system', 'content': initial_prompt},
                {'role': 'system', 'content': 'Final instruction on output format.'}
            ]

            full_initial_prompt_clean = ' '.join([item['content'] for item in full_initial_prompt])

            # Ask ChatGPT
            completion = client.chat.completions.create(
                model=model,
                messages=full_initial_prompt,
                seed=seed,
                max_tokens=max_tokens,
                temperature=temperature)
            response = completion.choices[0].message.content

            # Get the number of tokens used in the response
            num_tokens_used = completion.usage.total_tokens

            # Process the response and convert response to string if it's not
            if isinstance(response, str) and ':' in response:
                participant, message = response.split(":", 1)
                participant = participant.strip()
                message = message.strip()
            else:
                participant = 'System'
                message = str(response)

            # Include initial message in output
            initial_message = {
                "model": model,
                "seed": seed,
                "temperature": temperature,
                "max_tokens": max_tokens,
                "tokens_used": num_tokens_used,
                "unique_id": unique_id,
                "participant_1": 'Researcher',
                "initial_prompt": full_initial_prompt_clean,
                "participant_2": 'GPT',
                "prediction": message, # this is the output
                "variable_name": variable_name, # extra columns
                "num_tokens_used": num_tokens_used,
                "query_date": datetime.now().strftime("%Y-%m-%d"),
                "query_time": datetime.now().strftime("%H:%M:%S")
            }
            conversation_log.append(initial_message)

    # Create a DataFrame from the conversation log
    conversation_df = pd.DataFrame(conversation_log)
    # Drop duplicates
    conversation_df.drop_duplicates(inplace=True)

    # Append the conversation to a CSV file
    file_exists = os.path.isfile(csv_file)
    csv_filename = csv_file
    conversation_df.to_csv(csv_filename, mode='a', index=False, header=not file_exists)

    # Print the entire conversation if requested
    if yes_print == 'yes':
        for index, row in conversation_df.iterrows():
            print(f"Model: {row['model']}, Seed: {row['seed']}, Temperature: {row['temperature']}, Max tokens: {row['max_tokens']}, Tokens used: {row['tokens_used']}")
            print(f"unique_id: {row['unique_id']}")
            print(f"Participant 1: {row['participant_1']}")
            print(f"Initial Prompt: {row['initial_prompt']}")
            if row['participant_2'] is not None:
                print(f"participant_2: {row['participant_2']}")
            print(f"prediction: {row['prediction']}")
            print(f"variable_name: {row['variable_name']}")
            print(f"num_tokens_used: {row['num_tokens_used']}")
            print(f"query_time: {row['query_time']}\n")

    return conversation_df

In [ ]:
# All models
models = ["gpt-3.5-turbo-0125", "gpt-4o-mini"] 
# "gpt-4", "gpt-4o", "gpt-4-turbo", "gpt-4-1106-preview", "gpt-4-0613"

In [ ]:
# setting up the output csv
csv_file = "[PATH TO OUTPUT FILE].csv"

In [6]:
### bootstrap answers: ask each question 1 time(s)
# I do this since sometimes the model does not yield the same answer for the same exact prompt.
# I take the median response.

temperature = [1]

yes_print = "no"
loop_count = 1
    # Loop through each model

for temp in tqdm(temperature):
    for model in tqdm(models):
        print(f"Model: {model}")
        for _ in tqdm(range(loop_count)):
            print(f"Loop iteration: {_ + 1}")
            query_valuation(model, df, set_seed, temp, set_maxtokens, csv_file, "no")

NameError: name 'tqdm' is not defined